# 📘 Bài 4 — Hàm & File I/O

**Tuần 3**

## 1. Hàm là cách đặt tên cho một ý tưởng

In [ ]:
def tinh_bmi(can_nang: float, chieu_cao: float) -> float:
    """Tính chỉ số BMI.

    Args:
        can_nang: cân nặng (kg)
        chieu_cao: chiều cao (mét)
    Returns:
        Chỉ số BMI
    """
    return can_nang / (chieu_cao ** 2)


print(f"{tinh_bmi(60, 1.7):.1f}")
print(tinh_bmi.__doc__.splitlines()[0])

> Docstring không phải trang trí. Gõ `tinh_bmi?` trong notebook hoặc rê chuột lên tên hàm trong VSCode sẽ hiện nó ra.

## 2. Tham số

In [ ]:
def chao(ten: str, loi_chao: str = "Xin chào") -> str:
    return f"{loi_chao}, {ten}!"

print(chao("An"))
print(chao("An", "Hi"))
print(chao(loi_chao="Chào bạn", ten="An"))   # gọi bằng tên -> thứ tự không quan trọng

In [ ]:
def f(*args, **kwargs):
    print("args   =", args)
    print("kwargs =", kwargs)

f(1, 2, ten="An", tuoi=25)

In [ ]:
# Giải nén khi gọi hàm
def cong(a, b, c):
    return a + b + c

nums = [1, 2, 3]
cfg = {"a": 1, "b": 2, "c": 3}
print(cong(*nums))
print(cong(**cfg))

## 3. ⚠️ Bẫy mutable default argument

Bẫy kinh điển, và là câu hỏi phỏng vấn rất hay gặp.

In [ ]:
def sai(item, gio=[]):
    gio.append(item)
    return gio

print(sai("táo"))
print(sai("cam"))     # BẤT NGỜ: ['táo', 'cam']
print("Giá trị mặc định hiện tại:", sai.__defaults__)

**Vì sao?** Giá trị mặc định được tạo **một lần** khi Python đọc dòng `def`, không phải mỗi lần gọi. List đó tồn tại mãi và lớn dần.

In [ ]:
def dung(item, gio=None):
    if gio is None:
        gio = []           # tạo list MỚI mỗi lần gọi
    gio.append(item)
    return gio

print(dung("táo"))
print(dung("cam"))

## 4. Hàm là giá trị

In [ ]:
def nhan_doi(x):
    return x * 2

f = nhan_doi                       # gán hàm vào biến
print(f(5))

print(list(map(nhan_doi, [1, 2, 3])))
print([nhan_doi(x) for x in [1, 2, 3]])

# Truyền hàm làm tiêu chí sắp xếp
sp = [{"ten": "A", "gia": 300}, {"ten": "B", "gia": 100}]
print(sorted(sp, key=lambda s: s["gia"]))

> Ý này quay lại liên tục về sau: `df.apply(ham)` trong pandas, `key=` khi sắp xếp, decorator, callback.

## 5. Hàm thuần khiết (pure function)

In [ ]:
# ❌ Sửa dữ liệu gốc — người gọi không ngờ tới
def tang_gia_xau(sp_list, pt):
    for sp in sp_list:
        sp["gia"] = sp["gia"] * (1 + pt / 100)
    return sp_list

goc = [{"ten": "A", "gia": 100}]
tang_gia_xau(goc, 10)
print("Sau khi gọi, dữ liệu gốc:", goc)      # đã bị đổi!

In [ ]:
# ✅ Tạo dữ liệu mới
def tang_gia(sp_list, pt):
    return [{**sp, "gia": round(sp["gia"] * (1 + pt / 100))} for sp in sp_list]

goc = [{"ten": "A", "gia": 100}]
moi = tang_gia(goc, 10)
print("gốc:", goc)
print("mới:", moi)

## 6. Đường dẫn với pathlib

In [ ]:
from pathlib import Path

p = Path("data") / "raw" / "sales.csv"      # ✅ đúng trên mọi hệ điều hành
print(p)
print("tên         :", p.name)
print("không ext   :", p.stem)
print("phần mở rộng:", p.suffix)
print("thư mục cha :", p.parent)
print("tồn tại?    :", p.exists())

> Đừng viết `"data\sales.csv"` trong Python — `\` là ký tự thoát. `"data\new.csv"` chứa một dấu xuống dòng!

## 7. Đọc & ghi file

In [ ]:
from pathlib import Path
import tempfile, json, csv

tmp = Path(tempfile.mkdtemp())
print("Thư mục tạm:", tmp)

# --- Văn bản ---
with open(tmp / "a.txt", "w", encoding="utf-8") as f:
    f.write("Dòng 1\n")
    f.write("Dòng 2\n")

with open(tmp / "a.txt", encoding="utf-8") as f:
    print(repr(f.read()))

In [ ]:
# Đọc từng dòng — tiết kiệm bộ nhớ, file 10GB vẫn chạy được
with open(tmp / "a.txt", encoding="utf-8") as f:
    for i, dong in enumerate(f, 1):
        print(i, repr(dong.strip()))

In [ ]:
# --- JSON ---
du_lieu = {"ten": "Nguyễn Văn Nam", "diem": [8, 9]}

with open(tmp / "a.json", "w", encoding="utf-8") as f:
    json.dump(du_lieu, f, ensure_ascii=False, indent=2)

print((tmp / "a.json").read_text(encoding="utf-8"))

In [ ]:
# So sánh khi THIẾU ensure_ascii=False
print(json.dumps({"ten": "Nguyễn"}))                      # bị escape
print(json.dumps({"ten": "Nguyễn"}, ensure_ascii=False))  # giữ nguyên chữ

In [ ]:
# --- CSV ---
rows = [{"ten": "An", "diem": 8}, {"ten": "Bình", "diem": 9}]

with open(tmp / "a.csv", "w", encoding="utf-8", newline="") as f:   # newline="" BẮT BUỘC
    w = csv.DictWriter(f, fieldnames=["ten", "diem"])
    w.writeheader()
    w.writerows(rows)

print((tmp / "a.csv").read_text(encoding="utf-8"))

with open(tmp / "a.csv", encoding="utf-8", newline="") as f:
    doc_lai = list(csv.DictReader(f))
print(doc_lai)
print("Chú ý: mọi giá trị đều là CHUỖI ->", type(doc_lai[0]["diem"]).__name__)

### Ba điều phải nhớ về file

| Điều | Vì sao |
|---|---|
| Luôn dùng `with open(...)` | Tự đóng file kể cả khi có lỗi |
| Luôn `encoding="utf-8"` | Windows mặc định không phải UTF-8 → hỏng tiếng Việt |
| `newline=""` khi dùng module `csv` | Nếu không, Windows sinh dòng trống xen giữa |

## ✍️ Bài tập

```bash
pytest tests/phase01/test_ex04.py tests/phase01/test_ex05.py -v
```